In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark import pipelines as dp

In [0]:
@dp.table()
def gold_returns_stage():
    return spark.read.table('adventure_works.silver.returns')

In [0]:
@dp.materialized_view()
def returns_matt_view():
    df_returns= spark.read.table('Live.gold_returns_stage')
    df_calendar = spark.read.table('adventure_works.gold.dim_calendar')
    df_join = df_returns.join(df_calendar,df_returns['ReturnDate'] == df_calendar['Date'],how='left')
    df_final = df_join.select(col('DimCalendarKey').alias('ReturnDate'),col('TerritoryKey'),col('ProductKey'),col('ReturnQuantity'))

    return df_final

In [0]:
dp.create_streaming_table('dim_returns')

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7928176091554064>, line 1
----> 1 dp.create_streaming_table('dim_returns')

NameError: name 'dp' is not defined

In [0]:
dp.create_auto_cdc_flow(
    target = 'dim_returns',
    source = 'Live.returns_matt_view',
    keys = ['ReturnDate','TerritoryKey','ProductKey','ReturnQuantity'],
    sequence_by = col('ReturnDate'),
    stored_as_scd_type=1
)